# Build: Attention-Enhanced Seq2Seq

**Module 11** — The Attention Mechanism

> From the [Zylo Learning Platform](https://socrapy-local.vercel.app) Sequence Models course.


## Context

Let's add attention to a seq2seq model. The key change is in the decoder: at each step, instead of using only its hidden state, it also computes a context vector by attending to all encoder states.


## Setup

Install required packages (skip if already installed):


In [ ]:
!pip install torch -q


## Code

Run each cell in order:


In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class Attention(nn.Module):
    """Luong-style multiplicative attention."""
    def __init__(self, hidden_dim):
        super().__init__()
        self.W = nn.Linear(hidden_dim, hidden_dim, bias=False)
    
    def forward(self, decoder_state, encoder_states):
        """
        Args:
            decoder_state: [batch, hidden_dim]
            encoder_states: [batch, src_len, hidden_dim]
        Returns:
            context: [batch, hidden_dim]
            weights: [batch, src_len]
        """
        # Score: s^T W h for each encoder state
        # decoder_state: [B, H] -> [B, 1, H]
        query = self.W(decoder_state).unsqueeze(1)  # [B, 1, H]
        scores = torch.bmm(query, encoder_states.transpose(1, 2))  # [B, 1, src_len]
        scores = scores.squeeze(1)  # [B, src_len]
        
        weights = F.softmax(scores, dim=-1)  # [B, src_len]
        
        # Context: weighted sum of encoder states
        context = torch.bmm(weights.unsqueeze(1), encoder_states)  # [B, 1, H]
        context = context.squeeze(1)  # [B, H]
        
        return context, weights


class AttentionEncoder(nn.Module):
    def __init__(self, vocab_size, embed_dim, hidden_dim):
        super().__init__()
        self.embed = nn.Embedding(vocab_size, embed_dim)
        self.lstm = nn.LSTM(embed_dim, hidden_dim, batch_first=True)
    
    def forward(self, x):
        embedded = self.embed(x)
        outputs, (h, c) = self.lstm(embedded)
        return outputs, (h, c)  # Return ALL states, not just final


class AttentionDecoder(nn.Module):
    def __init__(self, vocab_size, embed_dim, hidden_dim):
        super().__init__()
        self.embed = nn.Embedding(vocab_size, embed_dim)
        self.lstm = nn.LSTM(embed_dim + hidden_dim, hidden_dim, batch_first=True)
        self.attention = Attention(hidden_dim)
        self.fc = nn.Linear(hidden_dim * 2, vocab_size)  # concat context + state
    
    def forward(self, x, h, c, encoder_states):
        embedded = self.embed(x.unsqueeze(1))  # [B, 1, E]
        
        # Attention context from previous step
        context, weights = self.attention(h.squeeze(0), encoder_states)
        
        # Concatenate embedding with context
        lstm_input = torch.cat([embedded, context.unsqueeze(1)], dim=2)  # [B, 1, E+H]
        output, (h, c) = self.lstm(lstm_input, (h, c))
        
        # Output: concat hidden state with new context
        new_context, weights = self.attention(h.squeeze(0), encoder_states)
        combined = torch.cat([h.squeeze(0), new_context], dim=1)  # [B, 2H]
        pred = self.fc(combined)  # [B, vocab_size]
        
        return pred, h, c, weights


class AttentionSeq2Seq(nn.Module):
    def __init__(self, encoder, decoder, device):
        super().__init__()
        self.encoder = encoder
        self.decoder = decoder
        self.device = device
    
    def forward(self, src, tgt, teacher_forcing_ratio=0.5):
        batch_size = src.shape[0]
        tgt_len = tgt.shape[1]
        vocab_size = self.decoder.fc.out_features
        
        outputs = torch.zeros(batch_size, tgt_len, vocab_size).to(self.device)
        all_weights = []
        
        encoder_states, (h, c) = self.encoder(src)
        inp = tgt[:, 0]
        
        for t in range(1, tgt_len):
            pred, h, c, weights = self.decoder(inp, h, c, encoder_states)
            outputs[:, t] = pred
            all_weights.append(weights.detach())
            
            if torch.rand(1).item() < teacher_forcing_ratio:
                inp = tgt[:, t]
            else:
                inp = pred.argmax(1)
        
        return outputs, all_weights

### Step 2


In [ ]:

# Usage
VOCAB_SIZE = 50
EMBED_DIM = 32
HIDDEN_DIM = 64
device = torch.device('cpu')

enc = AttentionEncoder(VOCAB_SIZE, EMBED_DIM, HIDDEN_DIM)
dec = AttentionDecoder(VOCAB_SIZE, EMBED_DIM, HIDDEN_DIM)
model = AttentionSeq2Seq(enc, dec, device)

print(f'Total parameters: {sum(p.numel() for p in model.parameters()):,}')
print(f'Attention adds: W matrix ({HIDDEN_DIM}x{HIDDEN_DIM} = {HIDDEN_DIM**2:,} params)')
print(f'\nKey difference from vanilla seq2seq:')
print(f'  - Encoder returns ALL hidden states (not just final)')
print(f'  - Decoder computes attention weights at each step')
print(f'  - Context vector is DIFFERENT at each decoder step')

## Key Takeaway

The attention weights are returned alongside the predictions. You can use them to visualize what the model is 'looking at' --- the alignment heatmaps from the previous section. This interpretability is a bonus you don't get from most neural network components.

---

*Generated from the [Zylo Sequence Models Course](https://socrapy-local.vercel.app). Continue learning at the platform for interactive exercises, quizzes, and AI tutoring.*
